# Lookup-only public catalog — anonymous demo

Shows the *lookup-only* access mode for a GeoID catalog:

- Anonymous callers **CAN**:
  - `POST /search/catalogs/{cat}/items-search` (body `{"geoid": ...}`) — fetch one item by geoid (resolved catalog-wide)
  - `POST /search/catalogs/{cat}/items-search` (body `{"external_id": ..., "collection_id": ...}`) — resolve by external id within a named collection (`collection_id` required; external_id is not globally unique)
  - `GET /{stac|features}/catalogs/{cat}/collections/{coll}/items/{id}` — exact-item GET
- Anonymous callers **CANNOT**:
  - List collections / items, run STAC search, or perform any write

This is the **read/lookup half** of the lookup-only story: it assumes a
catalog that has already been provisioned and opted into anonymous lookup.
For the operator-side setup (creating the catalog, applying the routing
preset, opting collections into anonymous create, and POSTing items as an
anonymous client), see the companion notebook
`uc_lookup_only_anonymous_write.ipynb` (web extension), which exercises the
full lookup-only-catalog + anonymous-write + geoid/external_id-retrieval
flow end to end.

Two endpoints used:
- `https://data.review.fao.org/geospatial/v2/api/catalog/` — anonymous demo target
- `https://data.review.fao.org/geospatial/v2/api/auth/` — same surface that requires Keycloak auth, used to prove the DENY actually fires when a caller attempts enumeration without a token.

Run from JupyterLite at `https://data.review.fao.org/geospatial/v2/api/catalog/web/#notebooks`.

In [ ]:
import json
import os
import time

import httpx
from dotenv import load_dotenv, find_dotenv
load_dotenv(find_dotenv(usecwd=True), override=False)

# Auto-provision DYNASTORE_TOKEN via IDP client_credentials if not already set
if not os.environ.get("DYNASTORE_TOKEN"):
    _idp = (os.environ.get("IDP_PUBLIC_URL") or os.environ.get("IDP_ISSUER_URL", "")).rstrip("/")
    _cid = os.environ.get("IDP_CLIENT_ID", "")
    _sec = os.environ.get("IDP_CLIENT_SECRET", "")
    if _idp and _cid and _sec:
        try:
            _r = httpx.post(
                f"{_idp}/protocol/openid-connect/token",
                data={"grant_type": "client_credentials", "client_id": _cid, "client_secret": _sec},
                timeout=10,
            )
            if _r.status_code == 200:
                os.environ["DYNASTORE_TOKEN"] = _r.json().get("access_token", "")
        except Exception:
            pass

CATALOG_BASE = os.environ.get(
    "DYNASTORE_CATALOG_BASE",
    os.environ.get("DYNASTORE_BASE_URL", "http://localhost:8080"),
)
AUTH_BASE = os.environ.get(
    "DYNASTORE_AUTH_BASE",
    os.environ.get("DYNASTORE_BASE_URL", "http://localhost:8080"),
)
PUB_CATALOG_ID = os.environ.get("PUB_CATALOG_ID", "public-demo")
PUB_COLL_ID = os.environ.get("PUB_COLL_ID", "items")

DEFAULT_HEADERS = {"Content-Type": "application/json", "Accept": "application/json"}
anon_catalog = httpx.Client(base_url=CATALOG_BASE, headers=DEFAULT_HEADERS, timeout=30.0)
anon_auth = httpx.Client(base_url=AUTH_BASE, headers=DEFAULT_HEADERS, timeout=30.0)
GEOID = None
print(f"catalog: {CATALOG_BASE}\nauth:    {AUTH_BASE}\ncat={PUB_CATALOG_ID} coll={PUB_COLL_ID}")

## 0. Wait for the catalog to be reachable

On a freshly-deployed review env, catalog tenant provisioning is async — a request issued before the schema is ready returns 404 even though the catalog will exist seconds later. Poll the catalog landing page until it answers, then continue with the lookup demo.

In [ ]:
MAX_WAIT_S = 30
POLL_S = 2
deadline = time.monotonic() + MAX_WAIT_S
ready = False
last_status = None
while time.monotonic() < deadline:
    r = anon_catalog.get(f"/stac/catalogs/{PUB_CATALOG_ID}")
    last_status = r.status_code
    # 200 = reachable; 401/403 = reachable but gated (still proves the
    # tenant is provisioned). Only 404/5xx mean keep waiting.
    if r.status_code in (200, 401, 403):
        ready = True
        break
    time.sleep(POLL_S)

if ready:
    print(f"Catalog {PUB_CATALOG_ID!r} reachable (HTTP {last_status}) — proceeding.")
else:
    raise RuntimeError(
        f"Catalog {PUB_CATALOG_ID!r} not reachable after {MAX_WAIT_S}s "
        f"(last status: {last_status}). Has the lookup-only demo catalog "
        f"been created and `catalog_lookup_audience.is_public=true` set?"
    )

## 1. Discover one geoid via external-id lookup

We don't know any geoids yet. POST an items-search with the *external* identifier we know — the catalog returns the matching geoid plus the collection it lives in.

In [ ]:
# Replace with any external_id known to your collection. The notebook tries
# a few common values; swap in something real for the demo.
EXTERNAL_ID = os.environ.get("EXTERNAL_ID", "pub-item-0-1")

r = anon_catalog.post(
    f"/search/catalogs/{PUB_CATALOG_ID}/items-search",
    json={"collection_id": PUB_COLL_ID, "external_id": EXTERNAL_ID, "limit": 1},
)
print(f"POST /search/.../items-search (by external_id): {r.status_code}")
GEOID = None
if r.is_success:
    body = r.json()
    print(json.dumps(body, indent=2)[:600])
    results = body.get("results") or []
    if results:
        GEOID = results[0].get("geoid") or results[0].get("id")
        print(f"\nResolved geoid: {GEOID!r}")
else:
    print(r.text[:400])

## 2. Fetch the same item by geoid

Same surface, resolving by geoid this time. Returns full PostGIS geometry — no ES simplification on this code path.

In [ ]:
if GEOID:
    r = anon_catalog.post(
        f"/search/catalogs/{PUB_CATALOG_ID}/items-search",
        json={"geoid": GEOID},
    )
    print(f"POST /search/.../items-search (by geoid={GEOID}): {r.status_code}")
    if r.is_success:
        body = r.json()
        results = body.get("results") or []
        if results:
            row = results[0]
            print(f"  geoid       : {row.get('geoid')}")
            print(f"  collection  : {row.get('collection_id')}")
            print(f"  external_id : {row.get('external_id')}")
            geom = row.get('geometry')
            if geom:
                print(f"  geometry    : {geom.get('type')} ({len(json.dumps(geom))} bytes)")
    else:
        print(r.text[:300])
else:
    print("(skipped — no geoid resolved in step 1)")

## 3. Exact-item GET via the OGC Features surface

Anonymous callers can also hit the canonical OGC route directly when the item id is already known. This is allowed by the per-protocol public-access policy and is *not* part of the enumeration DENY set.

In [ ]:
if GEOID:
    r = anon_catalog.get(
        f"/features/catalogs/{PUB_CATALOG_ID}/collections/{PUB_COLL_ID}/items/{GEOID}",
    )
    print(f"GET /features/.../items/{GEOID}: {r.status_code}")
    if r.is_success:
        body = r.json()
        print(f"  id      : {body.get('id')}")
        print(f"  type    : {body.get('type')}")
        print(f"  geometry: {(body.get('geometry') or {}).get('type')}")
    else:
        print(r.text[:300])
else:
    print("(skipped — no geoid resolved in step 1)")

## 4. Enumeration is blocked

Listing collections, listing items, and running an unconstrained search all require auth. We probe two surfaces:

- **catalog/** — IAM evaluates the DENY policy, returning 401/403.
- **auth/** — Keycloak short-circuits before the route runs; the practical effect is the same: no enumeration without identity.

In [ ]:
def probe(client, label, method, path, **kw):
    r = client.request(method, path, **kw)
    blocked = r.status_code in (401, 403)
    flag = "DENY  " if blocked else ("ALLOW " if r.is_success else "OTHER ")
    print(f"  {flag} {label:<20} {method:<5} {path:<70} -> {r.status_code}")
    return r

for client, label in [(anon_catalog, "catalog"), (anon_auth, "auth")]:
    print(f"--- {label} ---")
    probe(client, label, "GET",  f"/features/catalogs/{PUB_CATALOG_ID}/collections")
    probe(client, label, "GET",  f"/features/catalogs/{PUB_CATALOG_ID}/collections/{PUB_COLL_ID}/items")
    probe(client, label, "POST", f"/stac/catalogs/{PUB_CATALOG_ID}/search",
          content=json.dumps({"limit": 10}),
          headers={"Content-Type": "application/json"})
    probe(client, label, "GET",  f"/stac/catalogs/{PUB_CATALOG_ID}/collections")

## 5. Recap

| Action | Anonymous | Notes |
|---|---|---|
| `POST /search/catalogs/{cat}/items-search` (body `{"external_id": ..., "collection_id": ...}`) | ALLOW | Cell 1 (`collection_id` required) |
| `POST /search/catalogs/{cat}/items-search` (body `{"geoid": ...}`) | ALLOW | Cell 2 (catalog-wide) |
| `GET  /{stac|features}/catalogs/{cat}/collections/{coll}/items/{id}` | ALLOW | Cell 3 |
| `GET  /features/catalogs/{cat}/collections` | DENY | Cell 4 |
| `GET  .../items` (list) | DENY | Cell 4 |
| `POST /stac/catalogs/{cat}/search` | DENY | Cell 4 |
| Any write | DENY | not exercised |

Catalog opt-in: `PUT /configs/catalogs/{cat}/plugins/catalog_lookup_audience` with `{"is_public": true}`.

In [ ]:
anon_catalog.close()
anon_auth.close()